# HG4052 · Week 3 Practical
## Record it, break it, rebuild it

**No installs: librosa and every audio tool used here is preinstalled on Colab. Headphones on: this is the first notebook you can hear.**

By the end you will have:
- ✅ your recording (or the course fallback) loaded, plotted, and played back from raw numbers
- ✅ the downsampling safari ridden to the bottom: you heard the fricatives die, and Nyquist says why
- ✅ a vowel built from pure sine waves, one harmonic at a time: Fourier run in reverse
- ✅ the 3000 Hz impostor synthesized, and the folding rule that predicted it
- ✅ (take-home) your eight vowels on the Week 1 chart, next to Peterson & Barney's

**How this notebook works.** Same as Weeks 1 and 2: click a cell, press **Shift + Enter**, and read (now also hear) the output underneath. Cells marked **✏️ TODO** have one small blank to fill (always one line or less). Every function is provided for you to read and run, never to write.

**Short on time?** Prioritise **Setup → Part 2 → Part 3**. Part 1 is two cells on the way in; Part 4 is short; Part 5 is take-home by design.

---
### Before this notebook: the Praat block

This notebook is the Colab half of the practical. The Praat half comes first, cheat sheet on the slides:

- Record the hVd set ("heed, hid, head, had, hod, hawed, hood, who'd") and "She sees Sue's sheep" (New → Record mono Sound, 44100 Hz). No mic, or opting out? The fallback recordings downloaded below stand in wherever a recording of you is used.
- Flip one vowel between the two views: Spectrogram settings → window length 0.025 s (narrowband: count the harmonics), then 0.005 s (wideband: formants and glottal pulses). `data/week03_demo.wav`, the clean [ɑː] from the lecture, is there to practise on.
- Measure F1 and F2 at each vowel's midpoint (formant ceiling 5000 Hz for a typical male voice, 5500 Hz for a typical female voice) and write the eight pairs down. Part 5 needs them.
- Save your sentence recording as a WAV file (in the Objects window: Save → Save as WAV file) for the upload below.

---
## 0 · Setup

Week 1's commands first: where are you standing, and what is here?

In [ ]:
!pwd
!ls

**Imports.** Two new tools, both preinstalled on Colab: `librosa`, the standard Python audio library (loading, resampling, spectrograms), and `Audio`, the notebook's play button: give it an array of samples and a sampling rate, and it renders a player. `display(...)` shows more than one player per cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
from IPython.display import Audio, display

print("librosa", librosa.__version__, "loaded: the toolbox is open")

**Get today's audio.** Four short recordings from the course repo: the fallback sentence, two fallback hVd words, and the lecture's clean [ɑː]. All synthetic voices, made for this course. The cell checks its own work, file by file.

In [ ]:
!mkdir -p data
!wget -q -O data/sentence_sheep.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week03/sentence_sheep.wav
!wget -q -O data/hvd_heed.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week03/hvd_heed.wav
!wget -q -O data/hvd_hod.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week03/hvd_hod.wav
!wget -q -O data/week03_demo.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week03/week03_demo.wav

import os
all_ok = True
for name in ["sentence_sheep.wav", "hvd_heed.wav", "hvd_hod.wav", "week03_demo.wav"]:
    size = os.path.getsize(f"data/{name}") if os.path.exists(f"data/{name}") else 0
    ok = size > 10_000
    all_ok = all_ok and ok
    print(("✅" if ok else "❌"), f"{name:<20} {size:>8,} bytes")

if all_ok:
    print("\n✅ All four recordings are ready.")
else:
    print("\n❌ At least one download failed. The same files are on NTULearn in the Week 3")
    print("   folder: download them there, then drag each into data/ via the folder icon in")
    print("   Colab's left sidebar. Stuck? Ask on the NTULearn forum.")

**Your own recording (optional, encouraged).** If you recorded "She sees Sue's sheep" in the Praat block, upload it here and the whole notebook runs on your voice; otherwise it runs on the fallback. To upload, delete the leading `#` from the five commented lines and run the cell: a file dialog opens, and your file lands in `data/`.

In [ ]:
MY_RECORDING = "data/sentence_sheep.wav"        # the fallback: "She sees Sue's sheep."

# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, "data/my_recording.wav")
#     MY_RECORDING = "data/my_recording.wav"

print("Working file:", MY_RECORDING)

---
## 1 · Load and look

A WAV file is a short header (the lecture decoded one byte by byte) followed by a long list of numbers: the air-pressure snapshots. `librosa.load` reads it into a numpy array `y` and reports the sampling rate `sr`; `sr=None` means "keep the file's own rate, do not resample yet".

In [ ]:
y, sr = librosa.load(MY_RECORDING, sr=None)

print(f"{len(y):,} samples at {sr:,} Hz, every value between {y.min():.2f} and {y.max():.2f}")

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(np.arange(len(y)) / sr, y, linewidth=0.4)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude")
ax.set_title("the waveform: every number in y, drawn in order")
plt.show()

Audio(y, rate=sr)

In [ ]:
# ✏️ TODO: how long is the recording, in seconds?
# You have len(y) samples, and sr of them play per second. One division.
# Shape of the answer:   duration = len(y) / sr
duration = ...

if duration is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    print(f"duration = {duration:.2f} seconds")
    assert abs(duration - len(y) / sr) < 1e-9, "Expected samples divided by sampling rate"
    print("✅ Samples divided by rate: the first calculation of audio processing.")

---
## 2 · The downsampling safari

We resample the recording to 16000 Hz (this course's home rate), then ride down: 8000 (telephone land), 4000, 2000. Each rate hears nothing above its Nyquist ceiling, half the rate; `librosa.resample` removes what is above the ceiling *before* sampling, the lawful way.

**✏️ Commit before you run.** As the ceiling drops, which goes first: the sibilants (/ʃ/ and /s/ in "she sees Sue's sheep"), the vowel qualities, or the melody of the voice? Write your ranking, then run (double-click to edit):

> My prediction: **...**

In [ ]:
y16 = librosa.resample(y, orig_sr=sr, target_sr=16000)      # the reference version

versions = {16000: y16}
for rate in [8000, 4000, 2000]:
    versions[rate] = librosa.resample(y16, orig_sr=16000, target_sr=rate)

for rate, wave in versions.items():
    print(f"sampled at {rate:>6,} Hz: nothing above {rate // 2:,} Hz survives")
    display(Audio(wave, rate=rate))

**See what you heard.** A spectrogram is the lecture's recipe-unrolled-in-time: the provided `show_spectrogram` cuts the sound into 25 ms frames every 10 ms, takes the spectrum of each frame (`librosa.stft`, a row of dot products per column), and paints the columns side by side, loudness in dB as colour. All four panels share one frequency axis, so watch the ceiling descend.

In [ ]:
def show_spectrogram(wave, rate, title, ax, fmax=8000, win_s=0.025):
    """Frame the sound (win_s seconds every 10 ms), take a spectrum per frame, paint columns."""
    hop = int(0.010 * rate)
    mag = np.abs(librosa.stft(wave, n_fft=2048, win_length=int(win_s * rate), hop_length=hop))
    db = 20 * np.log10(mag + 1e-6)
    freqs = np.linspace(0, rate / 2, mag.shape[0])
    times = np.arange(mag.shape[1]) * hop / rate
    ax.pcolormesh(times, freqs, db, cmap="magma", vmin=db.max() - 60, vmax=db.max(), shading="auto")
    ax.set_ylim(0, fmax)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("frequency (Hz)")
    ax.set_title(title, fontsize=10)


fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for ax, (rate, wave) in zip(axes.flat, versions.items()):
    show_spectrogram(wave, rate, f"sampled at {rate:,} Hz: ceiling {rate // 2:,} Hz", ax)
plt.tight_layout()
plt.show()

**✏️ TODO (in words).** Which segments degraded first, and at which rate did each become unrecognisable? Explain with the Nyquist ceiling: name the frequencies each victim needed and no longer had. Was your prediction right? (Double-click to edit.)

> What died, when, and why: **...**

One phonetic consequence to carry away: telephone speech is the 8000 Hz row, which is why /s/ and /f/ are so easily confused on the phone.

---
## 3 · Build a voice from sinusoids

Fourier in reverse. The lecture stacked harmonics with buttons; those buttons are numpy one-liners, and now you own them. One pure tone first: y(t) = A sin(2πft), with f = 100 Hz.

In [ ]:
SR = 16000                # the synthesis world for this section: Nyquist 8000 Hz, plenty of room
DUR = 1.0
t = np.arange(int(SR * DUR)) / SR          # the time axis: sample number / rate = seconds

f0 = 100
harmonic1 = np.sin(2 * np.pi * f0 * t)     # the lecture's spinning wheel, one lap every 10 ms

Audio(harmonic1, rate=SR)

In [ ]:
# One new ingredient per line: each a pure sine at a multiple of f0, each quieter (amplitude 1/n).
harmonic2 = harmonic1 + (1 / 2) * np.sin(2 * np.pi * 2 * f0 * t)
harmonic3 = harmonic2 + (1 / 3) * np.sin(2 * np.pi * 3 * f0 * t)

print("100 + 200 Hz:")
display(Audio(harmonic2, rate=SR))
print("100 + 200 + 300 Hz (the lecture's three-sine buzz):")
display(Audio(harmonic3, rate=SR))

In [ ]:
def buzz(f0, n_harmonics, amps=None, sr=SR, dur=DUR):
    """Stack n_harmonics multiples of f0. amps[n - 1] scales harmonic n; default is 1/n."""
    t = np.arange(int(sr * dur)) / sr
    if amps is None:
        amps = [1 / n for n in range(1, n_harmonics + 1)]
    wave = np.zeros(len(t))
    for n in range(1, n_harmonics + 1):
        wave = wave + amps[n - 1] * np.sin(2 * np.pi * n * f0 * t)
    return wave / np.abs(wave).max()       # scale to a safe playback level


print("one harmonic (a bare hum):")
display(Audio(buzz(100, 1), rate=SR))
print("thirty harmonics (a glottal buzz: the source, no vocal tract yet):")
display(Audio(buzz(100, 30), rate=SR))

**Now the filter.** A buzz is not a vowel; a vowel is a buzz whose harmonics have been reshaped by the vocal tract's resonances. The provided `vowel_amps` builds a cardboard version of that filter: it keeps the 1/n source amplitudes but multiplies up any harmonic sitting near 730 Hz or 1090 Hz, the Peterson & Barney mean F1 and F2 of /ɑ/. Same source, two ways: listen for the vowel appearing.

In [ ]:
def vowel_amps(f0, n_harmonics, formants=(730, 1090), strength=6, width=120):
    """The /ɑ/ recipe: amplitude 1/n for harmonic n, boosted wherever it sits near a formant."""
    amps = []
    for n in range(1, n_harmonics + 1):
        freq = n * f0
        boost = 1.0
        for formant in formants:
            boost = boost + strength * np.exp(-((freq - formant) ** 2) / (2 * width ** 2))
        amps.append(boost / n)
    return amps


plain = buzz(100, 30)
ah = buzz(100, 30, amps=vowel_amps(100, 30))

print("the bare source (thirty harmonics at 1/n):")
display(Audio(plain, rate=SR))
print("the same source through the cardboard /ɑ/ filter (boosts near 730 and 1090 Hz):")
display(Audio(ah, rate=SR))

In [ ]:
# The same sound, seen both ways: what the ear receives vs the recipe we typed in.
fig, (ax_wave, ax_recipe) = plt.subplots(1, 2, figsize=(11, 3.4))

n_show = int(0.04 * SR)                    # the first 40 ms: four glottal periods at 100 Hz
ax_wave.plot(t[:n_show] * 1000, ah[:n_show], linewidth=0.9)
ax_wave.set_xlabel("time (ms)")
ax_wave.set_ylabel("amplitude")
ax_wave.set_title("waveform view: repeats every 10 ms, so F0 = 100 Hz")

amps = vowel_amps(100, 30)
ax_recipe.bar([n * 100 for n in range(1, 31)], amps, width=55, color="#4C72B0")
for formant in (730, 1090):
    ax_recipe.axvline(formant, color="#C44E52", linestyle="--", linewidth=1)
ax_recipe.set_xlabel("frequency (Hz)")
ax_recipe.set_ylabel("amplitude")
ax_recipe.set_title("recipe view: one bar per harmonic, humps at F1 and F2")

plt.tight_layout()
plt.show()

In [ ]:
# Did the cardboard filter get it right? The lecture's real [ɑː] (week03_demo.wav),
# averaged into a single spectrum: harmonic pickets, under humps at the formants.
demo, demo_sr = librosa.load("data/week03_demo.wav", sr=None)
mag = np.abs(librosa.stft(demo, n_fft=2048))
avg_db = 20 * np.log10(mag.mean(axis=1) + 1e-6)
freqs = np.linspace(0, demo_sr / 2, len(avg_db))

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(freqs, avg_db, linewidth=0.9)
for formant in (730, 1090):
    ax.axvline(formant, color="#C44E52", linestyle="--", linewidth=1)
ax.set_xlim(0, 3000)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("average level (dB)")
ax.set_title("the lecture's [ɑː]: pickets at multiples of its F0, humps near 730 and 1090 Hz")
plt.show()

In [ ]:
# ✏️ TODO: same vowel, different voice. Pick a new F0 between 50 and 400 Hz and listen.
# Shape of the answer:   F0_NEW = 150
F0_NEW = ...

if F0_NEW is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    assert 50 <= F0_NEW <= 400, "Keep F0 between 50 and 400 Hz so the harmonics stay in range"
    n_harm = int(3000 / F0_NEW)            # however many harmonics fit below 3000 Hz
    new_voice = buzz(F0_NEW, n_harm, amps=vowel_amps(F0_NEW, n_harm))
    print(f"F0 = {F0_NEW} Hz: harmonics every {F0_NEW} Hz, boosts still at 730 and 1090 Hz")
    display(Audio(new_voice, rate=SR))
    print("the 100 Hz original, for comparison:")
    display(Audio(ah, rate=SR))

**✏️ In words (double-click).** Between the two versions you just heard: what changed, and what stayed the same? Name which of the two belongs to the *source* and which to the *filter*.

> Changed: **...**
>
> Stayed: **...**

This is the source-filter model run forwards, and it is why a bass and a soprano can sing the same vowel.

---
## 4 · Hear aliasing

In the lecture, the wagon-wheel effect for sound: an undersampled frequency does not vanish, it reappears as a lower one. The recipe from the deck, now in your hands. Three routes to putting a 5000 Hz tone into an 8000 Hz world (Nyquist ceiling: 4000 Hz):

1. build the sine directly at 8000 samples per second, no filter, no mercy;
2. take an honest 48000 Hz recording of it and keep every 6th sample, again no filter;
3. the lawful route: `librosa.resample`, which filters everything above 4000 Hz away *first*.

**Predict before running:** which routes will sound identical, and what will the third one sound like?

In [ ]:
sr_fast = 48000                                   # an honest world: ceiling 24,000 Hz
t_fast = np.arange(sr_fast) / sr_fast             # one second
honest_5k = np.sin(2 * np.pi * 5000 * t_fast)
honest_3k = np.sin(2 * np.pi * 3000 * t_fast)
print("an honest 5000 Hz tone:")
display(Audio(honest_5k, rate=sr_fast, normalize=False))
print("an honest 3000 Hz tone (remember this sound):")
display(Audio(honest_3k, rate=sr_fast, normalize=False))

sr_low = 8000                                     # the crime scene: ceiling 4000 Hz
t_low = np.arange(sr_low) / sr_low
trapped = np.sin(2 * np.pi * 5000 * t_low)        # route 1: born undersampled
skimmed = honest_5k[::6]                          # route 2: 48000 / 6 = 8000, five of six samples binned
lawful = librosa.resample(honest_5k, orig_sr=sr_fast, target_sr=sr_low)   # route 3: filter, then sample
lawful = lawful[80:-80]                           # trim the filter's brief start-up click (10 ms per end)

print("route 1, the 5000 Hz sine built at 8000 samples per second:")
display(Audio(trapped, rate=sr_low, normalize=False))
print("route 2, the honest tone, five of every six samples thrown away:")
display(Audio(skimmed, rate=sr_low, normalize=False))
print(f"route 3, the lawful resampler (peak amplitude left: {np.abs(lawful).max():.7f}):")
display(Audio(np.clip(lawful, -1, 1), rate=sr_low, normalize=False))

**What you heard.** Routes 1 and 2 are the impostor: both play back at 3000 Hz, indistinguishable from the honest 3000 Hz tone, because 8000 samples per second of a 5000 Hz sine are *exactly* the samples of a 3000 Hz sine. Route 3 is near silence: the anti-aliasing filter removed the tone before sampling, so there was nothing left to lie about. Honest silence beats a confident impostor.

Now the sweep from the deck: a tone rising from 0 Hz aimed at 8000 Hz, in the same 8000 Hz world. It cannot get past the 4000 Hz ceiling, so listen to it bounce, and watch the spectrogram agree.

In [ ]:
sweep_dur = 4.0
target = 8000                                     # where the sweep thinks it is going
t_sweep = np.arange(int(sr_low * sweep_dur)) / sr_low
phase = 2 * np.pi * (target * t_sweep ** 2) / (2 * sweep_dur)   # frequency climbs 0 → 8000 Hz
sweep = np.sin(phase)
display(Audio(sweep, rate=sr_low, normalize=False))

fig, ax = plt.subplots(figsize=(9, 3.2))
show_spectrogram(sweep, sr_low, "a rising sweep in an 8000 Hz world: it bounces off the 4000 Hz ceiling", ax, fmax=4000)
plt.show()

In [ ]:
# ✏️ TODO: the folding rule, so you can predict an impostor instead of just hearing it.
# An undersampled frequency f, at sampling rate fs, reappears at   |f - fs * round(f / fs)|.
# Compute the alias of 5000 Hz at fs = 8000.
# Shape of the answer:   alias = abs(5000 - 8000 * round(5000 / 8000))
alias = ...

if alias is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    print(f"a 5000 Hz tone in an 8000 Hz world lands at {alias} Hz")
    assert alias == 3000, "Expected 3000: round(5000 / 8000) = 1, so |5000 - 8000| = 3000"
    print("✅ Exactly the impostor you heard. Cross-check: the lecture's EX 3.5,")
    print("   a 7000 Hz tone at fs = 10000, folds to", abs(7000 - 10000 * round(7000 / 10000)), "Hz.")

---
## 5 · Take-home: the formant round-trip

In Week 1 you plotted Peterson & Barney's 1952 measurements. Today you measured a voice yourself; the round-trip is to put your numbers on the same chart. First, a look at the two ends of the vowel space in the fallback voice, in the wideband view (the same 0.005 s window you used in Praat):

In [ ]:
heed, heed_sr = librosa.load("data/hvd_heed.wav", sr=None)
hod, hod_sr = librosa.load("data/hvd_hod.wav", sr=None)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
show_spectrogram(heed, heed_sr, "heed: F1 low, F2 high", ax1, fmax=4000, win_s=0.005)
show_spectrogram(hod, hod_sr, "hod: F1 high, F2 low, the two close together", ax2, fmax=4000, win_s=0.005)
plt.tight_layout()
plt.show()

In [ ]:
# ✏️ TODO: replace these numbers with YOUR eight pairs from the Praat block.
# Format: vowel: (F1, F2), both in Hz, measured at the vowel midpoint. Leave (None, None)
# for anything you could not measure; the plot skips it.
# No recording? Run as-is. The fallback values below were measured on the course fallback
# recordings (a synthetic voice) with the cheat-sheet recipe, formant ceiling 5500 Hz.
my_vowels = {
    "i": (480, 2780),      # heed
    "ɪ": (690, 2400),      # hid
    "ɛ": (940, 2180),      # head
    "æ": (1030, 2000),     # had
    "ɑ": (1060, 1370),     # hod
    "ɔ": (1060, 1190),     # hawed
    "ʊ": (None, None),     # hood (no fallback recording: fill only if you measured your own)
    "u": (540, 1300),      # who'd
}

PB_MEANS = {           # Peterson & Barney (1952) male-speaker means, exactly as in Week 1
    "i": (270, 2290), "ɪ": (390, 1990), "ɛ": (530, 1840), "æ": (660, 1720),
    "ɑ": (730, 1090), "ɔ": (570, 840), "ʊ": (440, 1020), "u": (300, 870),
}

measured = {v: pair for v, pair in my_vowels.items() if pair[0] is not None}
print(f"{len(measured)} of 8 vowels ready to plot")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.8))

for v, (f1, f2) in PB_MEANS.items():
    ax.scatter(f2, f1, color="#3E8E8E", alpha=0.55, s=45)
    ax.text(f2, f1 - 30, v, fontsize=15, ha="center", color="#3E8E8E")

for v, (f1, f2) in measured.items():
    ax.scatter(f2, f1, color="#C44E52", s=60)
    ax.text(f2, f1 + 55, v, fontsize=15, ha="center", color="#C44E52")

ax.invert_xaxis()                          # Week 1's conventions: back vowels to the right,
ax.invert_yaxis()                          # open vowels at the bottom
ax.set_xlabel("← F2 (Hz)")
ax.set_ylabel("← F1 (Hz)")
ax.set_title("the round-trip: your vowels (red) vs Peterson & Barney means (teal)")
plt.show()

**✏️ The take-home question (double-click, a short paragraph).** Where do your vowels sit relative to the 1952 American men, and why? For each big gap, decide: **measurement artifact** (wrong formant ceiling, a creaky stretch, the cursor off the midpoint, F1 and F2 merged by the tracker) or **sociophonetic reality** (your voice, your accent, seventy years of change; for many of you, Singapore English rather than mid-century American). Differences are findings, not errors. If you used the fallback values: they come from a synthetic voice, so which deviations do you trust?

> **...**

Attach this chart and your annotated narrowband/wideband screenshots to the take-home thread on NTULearn.

---
## 6 · Stretch: the KlattGrid playground (in Praat, seeds Week 10)

`vowel_amps` was a cardboard filter; Praat ships a real one, the Klatt synthesizer. Paste this into a Praat script window (Praat → New Praat script), then Run → Run:

```
Create KlattGrid: "vowel", 0, 1.0, 6, 1, 1, 6, 1, 1, 1
Add pitch point: 0, 100
Add voicing amplitude point: 0, 90
Add oral formant frequency point: 1, 0, 730
Add oral formant bandwidth point: 1, 0, 80
Add oral formant frequency point: 2, 0, 1090
Add oral formant bandwidth point: 2, 0, 90
To Sound
Play
```

A 100 Hz source through a real /ɑ/ filter. (Praat will warn about missing nasal and tracheal formants; ignore it.) Now the experiment: keep the source fixed, morph only the filter. Add these two lines just before `To Sound` and run again:

```
Add oral formant frequency point: 1, 1.0, 270
Add oral formant frequency point: 2, 1.0, 2290
```

The pitch never moves, yet the vowel slides /ɑ/ → /i/ over the second: the filter alone changes the vowel. Week 10's formant synthesizers are this object with a control panel.

### Appendix: the code behind the lecture's bit-depth clips

The other ruler from the lecture, quantization, in one rounding line. This is essentially how `bitdepth_16/8/4/3.wav` in the course repo were made (same idea; the repo clips use a slightly different level grid and their own source sentence). Needs `y16` from Part 2.

In [ ]:
def quantise(wave, bits):
    """Round every sample to the nearest of the 2**bits levels the ruler allows."""
    levels = 2 ** (bits - 1) - 1               # 16 bits: 32,767 rungs each side of zero; 3 bits: 3 each side (7 levels in all)
    return np.round(np.clip(wave, -1, 1) * levels) / levels


for bits in [16, 8, 4, 3]:
    print(f"{bits} bits ({2 ** bits:,} levels): rounding error is a noise floor about {6 * bits} dB down")
    display(Audio(quantise(y16, bits), rate=16000, normalize=False))

---
## ✅ Done looks like

- eight F1/F2 pairs written down in the Praat block and typed into Part 5
- you heard the fricatives die on the way down the safari, and Nyquist says why
- a buzz that turned into /ɑ/ when two harmonic neighbourhoods were boosted
- an impostor at 3000 Hz that the folding rule predicted before you heard it
- (take-home) the round-trip chart: your vowels next to Peterson & Barney's

**What you built.** Week 1 plotted other people's measurements; today you made the sounds themselves: loaded them, broke them lawfully (resampling) and unlawfully (aliasing), and assembled a vowel out of pure tones. The buzz-through-a-filter is the source-filter model run forwards; Week 10's synthesizers industrialise it. Next week the direction reverses: the machine takes your recordings apart into features.

**Before you go.** Exit ticket on NTULearn, one sentence: what is still muddiest from today?

**Next week:** from waveform to features: mel spectrograms, MFCCs, and what a recognizer actually reads.